In [21]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [22]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from rich import print
import uuid

In [39]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add
from typing import Literal
from langchain.agents.structured_output import ToolStrategy

In [24]:
model = ChatOpenAI(
    model_name="gpt-4.1-mini",
    temperature=0,
    openai_api_key=api_key
)

In [25]:
ToneType = Literal["formal", "informal", "friendly", "professional", "persuasive"]
LengthType = Literal["short", "medium", "long"]
LanguageType = Literal["English", "Spanish", "French", "German", "Japanese"]

In [26]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list, description="Contains the conversation.")
    topic: str | None = Field(default=None, description="The topic of the email.")
    sender_email: str | None = Field(default=None, description="The email of the sender.")
    recipient_email: str | None = Field(default=None, description="The email of the recipient.")
    subject: str | None = Field(default=None, description="The subject of the email.")
    email_body: str | None = Field(default=None, description="The body of the email.")

In [27]:
class MailSpecificationSchema(BaseModel):
    topic: str = Field(description="The topic or purpose of the email.")

    tone: str = Field(
        default="professional",
        description="Email tone: formal, informal, friendly, professional, or persuasive."
    )

    length: str = Field(
        default="medium",
        description="Email length: short, medium, or long."
    )

    language: str = Field(
        default="English",
        description="Email language: English, Spanish, French, German, or Japanese."
    )

In [ ]:
@tool(args_schema=MailSpecificationSchema)
def mail_specification(
    topic: str | None = None,
    tone: ToneType | None = None,
    length: LengthType | None = None,
    language: LanguageType | None = None,
) -> HumanMessage:
    """
    Collects email specifications and returns a structured HumanMessage.
    """

    DEFAULT_TONE = "professional"
    DEFAULT_LENGTH = "medium"
    DEFAULT_LANGUAGE = "English"

    ALLOWED_TONES = ToneType.__args__
    ALLOWED_LENGTHS = LengthType.__args__
    ALLOWED_LANGUAGES = LanguageType.__args__

    if not topic:
        topic = input("Enter email topic: ").strip() or "General inquiry"

    if not tone:
        user_tone = input("Enter tone: ").strip().lower()
        tone = user_tone if user_tone in ALLOWED_TONES else DEFAULT_TONE

    if not length:
        user_length = input("Enter length: ").strip().lower()
        length = user_length if user_length in ALLOWED_LENGTHS else DEFAULT_LENGTH

    if not language:
        user_language = input("Enter language: ").strip().title()
        language = user_language if user_language in ALLOWED_LANGUAGES else DEFAULT_LANGUAGE

    content = f"""
        Please compose an email following these specifications:
            - Topic: {topic}
            - Tone: {tone}
            - Length: {length}
            - Language: {language}
        Ensure the email strictly follows these specifications.
    """

    return HumanMessage(content=content)

In [ ]:
@tool(args_schema=MailSpecificationSchema)
def save_specifications(
    topic: str | None = None,
    tone: ToneType | None = None,
    length: LengthType | None = None,
    language: LanguageType | None = None,
    runtime: ToolRuntime = None
) -> Command:
    """ 
        This tool is used to save the current email draft. It saves the information regarding the 
        topic,tone, length, and language of the email.
    """

    DEFAULT_TONE = "professional"
    DEFAULT_LENGTH = "medium"
    DEFAULT_LANGUAGE = "English"
    
    ALLOWED_TONES = ToneType.__args__
    ALLOWED_LENGTHS = LengthType.__args__
    ALLOWED_LANGUAGES = LanguageType.__args__
    
    if not topic:
        topic = input("Enter email topic: ").strip() or "General inquiry or conversation"
    
    if not tone:
        user_tone = input("Enter tone: ").strip().lower()
        tone = user_tone if user_tone in ALLOWED_TONES else DEFAULT_TONE
    
    if not length:
        user_length = input("Enter length: ").strip().lower()
        length = user_length if user_length in ALLOWED_LENGTHS else DEFAULT_LENGTH
    
    if not language:
        user_language = input("Enter language: ").strip().title()
        language = user_language if user_language in ALLOWED_LANGUAGES else DEFAULT_LANGUAGE

    updates = {}

    if topic is not None:
        updates["topic"] = topic

    if tone is not None:
        updates["tone"] = tone

    if length is not None:
        updates["length"] = length

    if language is not None:
        updates["language"] = language

    updates["messages"] = [
        ToolMessage(
            content="Email specifications updated.",
            tool_call_id=runtime.tool_call_id
        )
    ]

    return Command(update=updates)

In [42]:
@tool
def get_mail_specification(runtime: ToolRuntime) -> HumanMessage:
    """
    Reads email specifications like topic, tone, length, and language from graph state
    and generates a structured HumanMessage.
    """

    state = runtime.state

    topic = state.get("topic", "General inquiry")
    tone = state.get("tone", "professional")
    length = state.get("length", "medium")
    language = state.get("language", "English")

    content = f"""
    Please compose an email following these specifications:

    - Topic: {topic}
    - Tone: {tone}
    - Length: {length}
    - Language: {language}

    Ensure the email strictly follows these specifications.
    """

    return HumanMessage(content=content)

NameError: name 'ToolRuntime' is not defined

In [34]:
@tool
def save_draft():
    """ 
        This tool is used to save the current email draft. It saves the information regarding the 
        recipient's email address, sender email address, subject, and body of the email.
    """
    pass

In [35]:
@tool
def send_mail():
    """AIMessage
        This tool is used to send the email to the recipient.
    """
    pass

In [ ]:
tools = [save_draft,send_mail,mail_specification]
tool_by_name = {tool.name: tool for tool in tools}

system_prompt = """
    You are a helpful assistant that helps the user to compose an email.
    You have the access of the tool  to save the infoemation regarding the recipient's email address, sender email address, subject, and body of the email. 
    Tou also have the access of the tool to send email to the recipient.
"""

In [37]:
model_with_tools = model.bind_tools(
    tools=tools,
    parallel_tool_calls=False,
    system_message=SystemMessage(content=system_prompt)
)

In [ ]:
def chat_node(state:State):

    while True:
        # user_input.
        user_query = input("User: Write your query here: ")
        if user_query.lower() == "exit":
            print("Exiting the chat.")
            break

        # Making the copy of the messages to avoid mutating the original state.
        messages = state.messages.copy()

        # Appending the system prompt only if messages are empty.
        if not messages:
            messages.append(SystemMessage(content=system_prompt))
            
        messages.append(HumanMessage(content=user_query))

        # invoking the model with the messages
        response = model_with_tools(messages)
        
        if not response.tool_calls:
            return response.content
        else:
            # Handling tool calls
            for tool_call in response.tool_calls:
                tool_name = tool_call.tool_name
                tool_args = tool_call.args

                if tool_name in tool_by_name:
                    tool_function = tool_by_name[tool_name]
                    tool_response = tool_function(**tool_args)
                    messages.append(ToolMessage(content=tool_response.content, tool_name=tool_name))
                
                    


In [38]:
def chat_with_model_node(state: State):

    while True:

        user_query = input("User: ")

        if user_query.lower() == "exit":
            print("Exiting chat.")
            break

        messages = state.messages.copy()

        if not messages:
            messages.append(SystemMessage(content=system_prompt))

        messages.append(HumanMessage(content=user_query))

        while True:

            # Invoke LLM
            response = model_with_tools.invoke(messages)

            # Append assistant response
            messages.append(response)

            # Check whether LLM wants to call a tool
            if not response.tool_calls:
                print("Assistant:", response.content)
                break

            # Execute only one tool call
            tool_call = response.tool_calls[0]

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]

            # Execute tool
            tool_function = tool_by_name[tool_name]
            tool_result = tool_function.invoke(tool_args)

            # Append tool response
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call_id
                )
            )

    return {"messages": messages}